# MiniProject2

GH Username: PiSaucer  
NetID: asaucer


In [1]:
# Package installations
# !python -m pip install -U python-woc pandas numpy

# Part 1: Create the project summary file from World of Code data

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# Create the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# API key version
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits
netID = 'asaucer'
assignments = pd.read_csv('net2prj.csv')
projects = assignments.loc[assignments['netID'] == netID, 'WoC'].tolist()
assert len(projects) == 10
list_df_commits = []
for prj in projects:
  # The WoC project names are already provided in net2prj.csv
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits, ignore_index=True).drop_duplicates(['project', 'sha1'])
# perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv', index=False)
print(df.groupby('project').size().to_string())
df.head(1)


project
anustg_optics_verification      582
atchem_atchem2                 2825
awslabs_gluon-ts              10958
locusrobotics_fuse             2313
netket_netket                 11928
qiskit_qiskit-vscode           1259
rexcardan_evil-dicom            305
sfilippone_psblas3             3921
sigven_pcgr                    1428
threeml_threeml                5360


,sha1,project
0,0031aa3a7fc3e9736b0c306345808be96ff7f9cc,qiskit_qiskit-vscode


In [4]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split the unique commit sha1s in chunks
sha1s = df['sha1'].drop_duplicates().tolist()
chunks = [sha1s[x:x+10] for x in range(0, len(sha1s), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk)

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch', chunk)
  res = {k: v[0] for k, v in res.items() if v}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
print(f"Retrieved {df_commit_data.shape[0]} of {df.shape[0]} project-commit rows")
df_commit_data.head(2)


  0%|          | 8/4088 [00:08<1:09:51,  1.03s/it]

Got Errors {'0da0c2497ba21496f32ee3ec241228f62c313606': 'Key 0da0c2497ba21496f32ee3ec241228f62c313606 not found in /da5_fast/All.sha1c/commit_13.tch'}


 39%|███▉      | 1607/4088 [27:34<42:49,  1.04s/it] 

Got Errors {'9d1f097ed3887bab1a1fe9081a6f4978c6f2ff97': 'Key 9d1f097ed3887bab1a1fe9081a6f4978c6f2ff97 not found in /da5_fast/All.sha1c/commit_29.tch'}


 43%|████▎     | 1771/4088 [30:24<39:56,  1.03s/it]

...

 99%|█████████▉| 4060/4088 [1:09:50<00:28,  1.03s/it]

Got Errors {'0b0a2608be399fd3c9263f05075e9865f890594c': 'Key 0b0a2608be399fd3c9263f05075e9865f890594c not found in /da5_fast/All.sha1c/commit_11.tch'}


100%|██████████| 4088/4088 [1:10:19<00:00,  1.03s/it]


Retrieved 40856 of 40879 project-commit rows


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0031aa3a7fc3e9736b0c306345808be96ff7f9cc,b703f8047ca7392e2fa3597e5c5e898f2b88e64c,[cca890ac3001e3366ad451a584555c324ca4af40],Yeray Darias <yeray.darias@ibm.com>,1530528084,+0200,Yeray Darias <yeray.darias@ibm.com>,1530528084,+0200,Fixed the readme file from client\n,0031aa3a7fc3e9736b0c306345808be96ff7f9cc,qiskit_qiskit-vscode
1,008a9d04eacb7ada6003bdd129cc83bb1b6df767,85696033a72a26f476d723283defb523b165053b,[b97b234b2e95888da45afd1d6d9a0e7c396e2352],Yeray Darias <yeray.darias@ibm.com>,1523441297,+0200,Yeray Darias <yeray.darias@ibm.com>,1523441297,+0200,Arguments tester to check the type of the argu...,008a9d04eacb7ada6003bdd129cc83bb1b6df767,qiskit_qiskit-vscode


In [5]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0031aa3a7fc3e9736b0c306345808be96ff7f9cc', 'tree': 'b703f8047ca7392e2fa3597e5c5e898f2b88e64c', 'parent': ['cca890ac3001e3366ad451a584555c324ca4af40'], 'author': 'Yeray Darias <yeray.darias@ibm.com>', 'author_time': 1530528084, 'author_tz': '+0200', 'committer': 'Yeray Darias <yeray.darias@ibm.com>', 'committer_time': 1530528084, 'committer_tz': '+0200', 'message': 'Fixed the readme file from client\n'}


In [6]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project_wocid, commit_sha1, author, time, commit message
# Use the merged dataframe so each commit keeps its assigned project.
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].copy()
dfinf.columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


In [7]:
# Check if it has the right content
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,qiskit_qiskit-vscode,0031aa3a7fc3e9736b0c306345808be96ff7f9cc,Yeray Darias <yeray.darias@ibm.com>,1530528084,Fixed the readme file from client\n


In [8]:
# Save all your projects in the same comma-separated file
# Write once with column headers so rerunning this cell does not append duplicate rows
dfinf.to_csv(netID+'_project_summary.csv', index=False, header=True)

## Part 1 WoC and GitHub summary

WoC statistics below are calculated from the retrieved commit records and GitHub statistics were manually retrieved from the GitHub website.

| Project                                                                        | WoC commits | WoC authors | WoC min time | WoC max time | GitHub stars | GitHub forks | GitHub last commit date |
| ------------------------------------------------------------------------------ | ----------: | ----------: | -----------: | -----------: | -----------: | -----------: | ----------------------- |
| [qiskit_qiskit-vscode](https://github.com/Qiskit/qiskit-code-assistant-vscode) |       1,258 |          25 |   1519373836 |   1762282079 |           95 |           35 | 2026-05-29              |
| [awslabs_gluon-ts](https://github.com/awslabs/gluonts)                         |      10,958 |         294 |   1558011255 |   1757582731 |        5,244 |          838 | 2026-07-31              |
| [sigven_pcgr](https://github.com/sigven/pcgr)                                  |       1,428 |          19 |   1490040443 |   1759841533 |          284 |           52 | 2026-09-20              |
| [sfilippone_psblas3](https://github.com/sfilippone/psblas3)                    |       3,920 |          35 |    977932376 |   1761917630 |           70 |           20 | 2026-09-24              |
| [locusrobotics_fuse](https://github.com/locusrobotics/fuse)                    |       2,309 |          60 |   1530462104 |   1760966698 |          882 |          143 | 2026-08-19              |
| [threeml_threeml](https://github.com/threeML/threeML)                          |       5,360 |          81 |   1416590422 |   1761867478 |           86 |           70 | 2026-09-04              |
| [netket_netket](https://github.com/netket/netket)                              |      11,915 |         163 |   1524509351 |   1761145795 |          700 |          217 | 2026-09-24              |
| [anustg_optics_verification](https://github.com/anustg/optics-verification)    |         582 |          14 |   1237487527 |   1747229518 |            1 |            2 | 2025-05-14              |
| [atchem_atchem2](https://github.com/AtChem/AtChem2)                            |       2,822 |          20 |   1489401650 |   1761126665 |           72 |           25 | 2026-02-03              |
| [rexcardan_evil-dicom](https://github.com/rexcardan/Evil-DICOM)                |         304 |          35 |   1344434229 |   1726568961 |          192 |           96 | 2024-08-20              |